# Train YOLO-seg on Kaggle — 4-fold CV + fixed-test evaluation

Runs the Ultralytics YOLO instance-segmentation baselines (**yolo11** = proven,
**yolo26** = latest) under the *same* protocol as TransUNet: 4-fold CV, a fixed
held-out test set, and per-class / per-position / per-tooth-type **IoU + Dice**.
YOLO predicts one mask per tooth; evaluation rasterizes those into a 33-class
label map and reuses the dense-segmenter scoring code, so each fold's
`test_summary.json` is directly comparable to TransUNet's.

Like TransUNet, YOLO-seg is **image-only** — it needs only `data/splits`, the
same dataset you uploaded for TransUNet.

## Before you run
1. **Accelerator:** Settings → **GPU T4 x2** or **GPU P100**.
2. **Internet:** Settings → **On** (for `git`, `pip`, weight download).
3. **Data:** right panel → *Add Input* → your **pbl4-splits** dataset.

Then **Run All**, top to bottom. Re-running is safe: each cell re-establishes
its own state, and training skips folds that already finished.

## To download results afterwards
Files in `/kaggle/working` only appear in the **Output tab** after you save a
version. Use **Save Version → Quick Save** (with *Save output* on) — it snapshots
the current outputs **without** re-running. Or grab the zip directly from the
link printed by the last cell. (Do **not** use *Save & Run All* just to download —
it retrains from scratch.)

## 1. Configure

In [ ]:
REPO_URL = "https://github.com/Huay0804/PBL4.git"  # private? https://<TOKEN>@github.com/...
REPO_DIR = "/kaggle/working/PBL4"

MODELS = ["yolo11", "yolo26"]  # set to ["yolo11"] for just the proven baseline
SIZE   = "m"                    # n/s/m/l/x

## 2. Get the latest code and install Ultralytics
Clones if missing, then **always fast-forwards to `origin/main`** so you never
run stale scripts. `git reset --hard` only touches tracked files — your `runs/`
and `data/` outputs are left alone.

In [ ]:
import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("code:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
import torch, ultralytics
print(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | CUDA {torch.cuda.is_available()}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — turn on the accelerator (Settings).")

## 3. Wire up data and materialize the YOLO-seg dataset
`ensure_splits()` idempotently links `data/splits` to the mounted dataset (auto-
detected under `/kaggle/input`). It's re-defined in each data-touching cell so any
cell can be run on its own after a kernel restart.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def ensure_splits():
    """Idempotently link data/splits -> the mounted dataset. Safe to re-run."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input — add the "
                                "pbl4-splits dataset (right panel -> Add Input).")
    sp = os.path.dirname(hits[0])
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link):
        os.remove(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    elif os.path.exists(link):
        os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)

ensure_splits()
rc = subprocess.run([sys.executable, "scripts/prepare_yolo_seg_data.py"]).returncode
if rc != 0:
    raise SystemExit("Dataset materialization failed.")

## 4. Train all models × 4 folds
yolo11 trains at batch 8; **yolo26 uses auto-batch** (the script probes the GPU and
prints the size it chose, avoiding the OOM yolo26 hits at batch 8). Checkpoints land
at `runs/cv/fold_<k>/<model>_seg/weights/best.pt`. **Folds already trained are
skipped**, so interrupted runs resume cheaply.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)

def ensure_splits():
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("Add the pbl4-splits dataset (right panel -> Add Input).")
    sp = os.path.dirname(hits[0]); os.makedirs("data", exist_ok=True); link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link); print("linked data/splits ->", sp)

ensure_splits()
for model in MODELS:
    run_name = f"{model}_seg"
    for k in range(4):
        best = f"runs/cv/fold_{k}/{run_name}/weights/best.pt"
        if os.path.exists(best):
            print(f"skip {model} fold {k} (already trained)"); continue
        print(f"\n========== TRAIN {model} fold {k} ==========", flush=True)
        rc = subprocess.run(
            f"python -u scripts/train_yolo_seg_cv.py --model {model} --size {SIZE} "
            f"--fold {k} --skip-prepare",
            shell=True,
        ).returncode
        if rc != 0:
            raise SystemExit(f"{model} fold {k} training failed (exit {rc}).")
print("\nAll folds present.")

## 5. Evaluate each fold on the fixed test set
Writes `test_summary.json`, `test_metrics.json`, `per_class_*`, `per_position_*`,
`per_tooth_type_*` and `per_quadrant_*` next to each checkpoint — the same format
as the dense segmenters.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)

def ensure_splits():
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("Add the pbl4-splits dataset (right panel -> Add Input).")
    sp = os.path.dirname(hits[0]); os.makedirs("data", exist_ok=True); link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link); print("linked data/splits ->", sp)

ensure_splits()
for model in MODELS:
    for k in range(4):
        print(f"\n========== EVAL {model} fold {k} ==========", flush=True)
        rc = subprocess.run(
            f"python -u scripts/evaluate_yolo_seg.py --model {model} --size {SIZE} --cv-fold {k}",
            shell=True,
        ).returncode
        if rc != 0:
            raise SystemExit(f"{model} fold {k} evaluation failed (exit {rc}).")
print("\nAll evaluations done.")

## 6. Results — cross-validation aggregate per model
Mean ± std of the fixed-test macro/weighted IoU & Dice across the 4 folds, ready
to sit next to TransUNet's numbers.

In [ ]:
import glob, json, os
import numpy as np
os.chdir(REPO_DIR)

KEYS = ["macro_iou", "macro_dice", "weighted_iou", "weighted_dice"]
for model in MODELS:
    run_name = f"{model}_seg"
    summaries = sorted(glob.glob(f"runs/cv/fold_*/{run_name}/test_summary.json"))
    if not summaries:
        print(f"[{model}] no test summaries found"); continue
    rows = [json.load(open(p)) for p in summaries]
    print(f"\n=== {run_name}: {len(rows)} folds ===")
    agg = {}
    for key in KEYS:
        vals = np.array([r[key] for r in rows], float)
        agg[key] = {"mean": float(vals.mean()), "std": float(vals.std())}
        print(f"  {key:14s}: {vals.mean():.4f} ± {vals.std():.4f}")
    out = f"runs/cv/{run_name}_cv_summary.json"
    json.dump({"model": model, "folds": len(rows), "aggregate": agg,
               "per_fold": rows}, open(out, "w"), indent=2)
    print(f"  wrote {out}")

## 7. Package results for download
Writes **two** zips to `/kaggle/working`:
- `yolo_seg_models.zip` — just the `best.pt` checkpoints (the heavy files).
- `yolo_seg_results.zip` — all metrics, CV summaries, plots and CSVs (no weights).

**Downloading on Kaggle:** `FileLink` 404s in interactive sessions, so use the
editor's **right panel → Output → `/kaggle/working`** (download icon next to each
zip), or **Save Version → Quick Save** (with *Save output* on) and download from
the saved version's **Output tab**.

In [ ]:
import os, glob, subprocess
os.chdir(REPO_DIR)

models_zip  = "/kaggle/working/yolo_seg_models.zip"
results_zip = "/kaggle/working/yolo_seg_results.zip"
for z in (models_zip, results_zip):
    if os.path.exists(z):
        os.remove(z)

best_files = sorted(glob.glob("runs/cv/fold_*/*/weights/best.pt"))
if not best_files:
    raise SystemExit("No best.pt under runs/cv — run training (and eval) first.")

# Models: only the best.pt checkpoints (folder structure preserved).
subprocess.run(["zip", "-q", models_zip, *best_files], check=True)
# Results: everything under runs/cv except the weights dir.
subprocess.run(["zip", "-r", "-q", results_zip, "runs/cv", "-x", "*/weights/*"], check=True)

print(f"models  : {models_zip}  ({os.path.getsize(models_zip)/1e6:.1f} MB, {len(best_files)} best.pt)")
print(f"results : {results_zip} ({os.path.getsize(results_zip)/1e6:.1f} MB)")
print("\nDownload (FileLink 404s on Kaggle — use one of these):")
print("  - right panel -> Output -> /kaggle/working -> download icon next to the zip")
print("  - or Save Version -> Quick Save (Save output) -> version's Output tab -> Download")